<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/TOPO_CBP0_MODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Topo-CBP TRAIN

In [1]:
import random
import numpy as np
import hashlib
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# ==================== DETERMINISTIC SEED 123 LOCK ====================
SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class TopoGovernedQwen(nn.Module):
    """
    Wraps a Qwen transformer backbone with Topo-CBP topological governance,
    locking prime-indexed embedding weights against catastrophic forgetting.
    """
    def __init__(self, base_model, prime_indices=[2, 3, 5, 7, 11, 13]):
        super().__init__()
        self.model = base_model
        self.prime_indices = torch.tensor(prime_indices, dtype=torch.long)
        self.lambda_const = 0.9785142874

        # Capture anchor bedrock from input embedding weight matrix
        embed_weight = self.model.get_input_embeddings().weight
        with torch.no_grad():
            self.register_buffer("anchor_bedrock", embed_weight[self.prime_indices].clone())

    def pre_optimizer_step(self):
        """Phase 1: Project gradients to zero across invariant prime anchors."""
        embed_grad = self.model.get_input_embeddings().weight.grad
        if embed_grad is not None:
            embed_grad[self.prime_indices] = 0.0

    def post_optimizer_step(self):
        """Phase 2: Bitwise restoration against weight decay drift."""
        with torch.no_grad():
            embed_weight = self.model.get_input_embeddings().weight
            embed_weight[self.prime_indices] = self.anchor_bedrock

    def compute_anchor_hash(self) -> str:
        """Returns the SHA-256 state hash of the prime coordinate ring."""
        embed_weight = self.model.get_input_embeddings().weight
        # Cast to float32 to safely handle native BFloat16 tensors with NumPy
        raw_bytes = embed_weight[self.prime_indices].detach().to(torch.float32).cpu().contiguous().numpy().tobytes()
        return hashlib.sha256(raw_bytes).hexdigest()[:16]

    def forward(self, input_ids, labels=None):
        return self.model(input_ids=input_ids, labels=labels)

# --- Execution Pipeline ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INIT] Using device: {device} | Seed: {SEED}")

model_id = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
governed_model = TopoGovernedQwen(base_model).to(device)

optimizer = torch.optim.AdamW(governed_model.parameters(), lr=2e-5, weight_decay=1e-4)

initial_hash = governed_model.compute_anchor_hash()
print(f"[INIT] Locked Anchor Hash: {initial_hash}")

# Stream dataset corpus using fully-qualified path to avoid HfUriError
dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train", streaming=True)
streaming_iter = iter(dataset)

governed_model.train()
for step in range(5):
    # Fetch next valid non-empty sample
    sample = next(streaming_iter)
    while not sample["text"].strip():
        sample = next(streaming_iter)

    inputs = tokenizer(sample["text"], return_tensors="pt", max_length=256, truncation=True, padding="max_length").to(device)
    input_ids = inputs["input_ids"]

    # Forward & Backward Pass
    outputs = governed_model(input_ids=input_ids, labels=input_ids)
    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()

    # 1. Dual-Phase Topological Control (Pre-optimizer gradient zeroing)
    governed_model.pre_optimizer_step()

    optimizer.step()

    # 2. Dual-Phase Topological Control (Post-optimizer bedrock clamping)
    governed_model.post_optimizer_step()

    # Verification Telemetry
    with torch.no_grad():
        current_anchors = governed_model.model.get_input_embeddings().weight[governed_model.prime_indices]
        drift = torch.norm(current_anchors - governed_model.anchor_bedrock).item()

    current_hash = governed_model.compute_anchor_hash()
    status = "LOCKED" if (current_hash == initial_hash and drift == 0.0) else "CORRUPTED"

    print(f"[STEP {step+1}] Loss: {loss.item():.4f} | Hash: {current_hash} | Status: {status} | Drift: {drift:.10e}")

# --- Save Model & Topological Bedrock ---
save_directory = "./qwen2.5-0.5b-topo-governed-checkpoint"
governed_model.model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

torch.save({
    "anchor_bedrock": governed_model.anchor_bedrock,
    "prime_indices": governed_model.prime_indices,
    "initial_hash": initial_hash,
    "seed": SEED
}, f"{save_directory}/topo_bedrock_state.pt")

print(f"[SAVED] Checkpoint successfully written to {save_directory}")

[INIT] Using device: cuda | Seed: 123


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[INIT] Locked Anchor Hash: eb49aaf4e51888e1
[STEP 1] Loss: 11.7386 | Hash: eb49aaf4e51888e1 | Status: LOCKED | Drift: 0.0000000000e+00
[STEP 2] Loss: 4.4424 | Hash: eb49aaf4e51888e1 | Status: LOCKED | Drift: 0.0000000000e+00
[STEP 3] Loss: 2.1861 | Hash: eb49aaf4e51888e1 | Status: LOCKED | Drift: 0.0000000000e+00
[STEP 4] Loss: 1.6390 | Hash: eb49aaf4e51888e1 | Status: LOCKED | Drift: 0.0000000000e+00
[STEP 5] Loss: 0.3099 | Hash: eb49aaf4e51888e1 | Status: LOCKED | Drift: 0.0000000000e+00


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[SAVED] Checkpoint successfully written to ./qwen2.5-0.5b-topo-governed-checkpoint


## Topo-CBP: INFERENCE

In [2]:
import hashlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Load Checkpoint ---
save_directory = "./qwen2.5-0.5b-topo-governed-checkpoint"
print(f"[LOAD] Loading governed model from {save_directory}...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(save_directory)
model = AutoModelForCausalLM.from_pretrained(save_directory).to(device)

# Load saved topological state and metadata
bedrock_state = torch.load(f"{save_directory}/topo_bedrock_state.pt", map_location=device)
anchor_bedrock = bedrock_state["anchor_bedrock"]
prime_indices = bedrock_state["prime_indices"]
expected_hash = bedrock_state["initial_hash"]

# --- Verify Invariant Bedrock & Hash ---
embed_weight = model.get_input_embeddings().weight
current_anchors = embed_weight[prime_indices]
drift = torch.norm(current_anchors - anchor_bedrock).item()

raw_bytes = current_anchors.detach().to(torch.float32).cpu().contiguous().numpy().tobytes()
current_hash = hashlib.sha256(raw_bytes).hexdigest()[:16]

status = "LOCKED" if (current_hash == expected_hash and drift == 0.0) else "CORRUPTED"
print(f"[VERIFY] Loaded Anchor Hash: {current_hash} (Expected: {expected_hash})")
print(f"[VERIFY] Structural Status: {status} | Measured Drift: {drift:.10e}")

assert status == "LOCKED", f"Safety Interlock Tripped on Load! Drift: {drift:.10e}"

# --- Run Inference Test ---
prompt = "The architecture of permanence in neural networks relies on"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

model.eval()
with torch.no_grad():
    outputs = model.generate(
        inputs["input_ids"],
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n[INFERENCE RESULT]")
print("-" * 50)
print(generated_text)
print("-" * 50)

[LOAD] Loading governed model from ./qwen2.5-0.5b-topo-governed-checkpoint...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[VERIFY] Loaded Anchor Hash: eb49aaf4e51888e1 (Expected: eb49aaf4e51888e1)
[VERIFY] Structural Status: LOCKED | Measured Drift: 0.0000000000e+00

[INFERENCE RESULT]
--------------------------------------------------
The architecture of permanence in neural networks relies on a key principle, which is the presence of a "local memory" structure. This means that each neuron in a neural network has a local memory where it stores information about the inputs it has received. Consider a neural network with 4 neurons in a
--------------------------------------------------


## Topo-CBP: HF UPLOAD

In [3]:
from google.colab import userdata
from huggingface_hub import HfApi, login, create_repo
import torch

# Retrieve your token from Colab Secrets and authenticate
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

user_or_name = 'frankmorales2020'
model_name = 'qwen2.5-0.5b-topo-governed-cbp-wikitext'  # Unique name for this run
repo_id = f"{user_or_name}/{model_name}"
save_directory = "./qwen2.5-0.5b-topo-governed-checkpoint"

print(f"[HF] Initializing upload to repository: {repo_id}")

# Create repo if it doesn't exist
api = HfApi()
try:
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"[HF] Repository {repo_id} ready.")
except Exception as e:
    print(f"[HF] Notice during repo creation: {e}")

# Upload the entire folder including weights, tokenizer, and topological state
api.upload_folder(
    folder_path=save_directory,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Upload Qwen2.5-0.5B Topo-CBP governed checkpoint with WikiText training stream and verified invariant bedrock"
)

print(f"[SUCCESS] Model successfully uploaded to https://huggingface.co/{repo_id}")

[HF] Initializing upload to repository: frankmorales2020/qwen2.5-0.5b-topo-governed-cbp-wikitext
[HF] Repository frankmorales2020/qwen2.5-0.5b-topo-governed-cbp-wikitext ready.
[SUCCESS] Model successfully uploaded to https://huggingface.co/frankmorales2020/qwen2.5-0.5b-topo-governed-cbp-wikitext


## Topo-CBP: INFERENCE-HF

In [1]:
import hashlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download

repo_id = "frankmorales2020/qwen2.5-0.5b-topo-governed-cbp-wikitext"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[LOAD] Pulling governed model and bedrock from HF: {repo_id}")

# Load Tokenizer & Model directly from the Hub
tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModelForCausalLM.from_pretrained(repo_id).to(device)

# Download the saved topological bedrock state file from your repository
bedrock_path = hf_hub_download(repo_id=repo_id, filename="topo_bedrock_state.pt")
bedrock_state = torch.load(bedrock_path, map_location=device)

anchor_bedrock = bedrock_state["anchor_bedrock"]
prime_indices = bedrock_state["prime_indices"]
expected_hash = bedrock_state["initial_hash"]

# --- Verify Invariant Bedrock & Hash ---
embed_weight = model.get_input_embeddings().weight
current_anchors = embed_weight[prime_indices]
drift = torch.norm(current_anchors - anchor_bedrock).item()

raw_bytes = current_anchors.detach().to(torch.float32).cpu().contiguous().numpy().tobytes()
current_hash = hashlib.sha256(raw_bytes).hexdigest()[:16]

status = "LOCKED" if (current_hash == expected_hash and drift == 0.0) else "CORRUPTED"
print(f"[VERIFY] HF Hub Anchor Hash: {current_hash} (Expected: {expected_hash})")
print(f"[VERIFY] Structural Status: {status} | Measured Drift: {drift:.10e}")

assert status == "LOCKED", f"Safety Interlock Tripped! Drift: {drift:.10e}"

# --- Run Inference Test ---
prompt = "The architecture of permanence in neural networks relies on"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

model.eval()
with torch.no_grad():
    outputs = model.generate(
        inputs["input_ids"],
        max_new_tokens=50,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n[INFERENCE RESULT]")
print("-" * 50)
print(generated_text)
print("-" * 50)

[LOAD] Pulling governed model and bedrock from HF: frankmorales2020/qwen2.5-0.5b-topo-governed-cbp-wikitext


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[VERIFY] HF Hub Anchor Hash: eb49aaf4e51888e1 (Expected: eb49aaf4e51888e1)
[VERIFY] Structural Status: LOCKED | Measured Drift: 0.0000000000e+00

[INFERENCE RESULT]
--------------------------------------------------
The architecture of permanence in neural networks relies on a simple mathematical function called the sigmoid function. This function is a way of modeling a continuous signal that is typically represented by a straight line. The sigmoid function, represented by the equation \( \sigma(x) = \frac{1}{1 + e
--------------------------------------------------


In [2]:
import hashlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download

repo_id = "frankmorales2020/qwen2.5-0.5b-topo-governed-cbp-wikitext"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(repo_id)
model = AutoModelForCausalLM.from_pretrained(repo_id).to(device)

# Load topological bedrock state
bedrock_path = hf_hub_download(repo_id=repo_id, filename="topo_bedrock_state.pt")
bedrock_state = torch.load(bedrock_path, map_location=device)

anchor_bedrock = bedrock_state["anchor_bedrock"]
prime_indices = bedrock_state["prime_indices"]
expected_hash = bedrock_state["initial_hash"]

# Verify structural interlock
embed_weight = model.get_input_embeddings().weight
current_anchors = embed_weight[prime_indices]
drift = torch.norm(current_anchors - anchor_bedrock).item()

raw_bytes = current_anchors.detach().to(torch.float32).cpu().contiguous().numpy().tobytes()
current_hash = hashlib.sha256(raw_bytes).hexdigest()[:16]

print(f"Hash Match: {current_hash == expected_hash} | Measured Drift: {drift:.10e}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Hash Match: True | Measured Drift: 0.0000000000e+00
